# 22 — 均线策略与参数优化

**阶段**：第3阶段 · 策略开发与回测  
**模块**：模块3.1 策略实现  
**学习目标**：
- 理解双均线策略的金叉/死叉逻辑
- 掌握 vectorbt 向量化回测框架
- 学会参数网格搜索与夏普比热力图
- 理解样本内/样本外划分的必要性
- 识别参数优化的过拟合陷阱

**环境依赖**：`vectorbt`, `akshare`, `numpy`, `pandas`, `matplotlib`


## 0. 环境与数据准备

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import vectorbt as vbt

# 中文显示
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti SC', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print(f"vectorbt: {vbt.__version__}")
print(f"numpy:    {np.__version__}")
print(f"pandas:   {pd.__version__}")


### 获取沪深300数据

In [ ]:
import akshare as ak

try:
    df_raw = ak.stock_zh_index_daily(symbol="sh000300")
    df_raw.columns = [c.lower() for c in df_raw.columns]
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    df_raw = df_raw.set_index('date').sort_index()
    # 选取 close 列（akshare 中通常为 'close'）
    price_col = 'close' if 'close' in df_raw.columns else df_raw.columns[3]
    print(f"数据范围: {df_raw.index.min().date()} ~ {df_raw.index.max().date()}")
    print(f"数据行数: {len(df_raw)}")
    print(f"价格列名: {price_col}")
    df_raw[[price_col]].tail()
except Exception as e:
    print(f"AKShare 获取失败: {e}")
    print("使用模拟数据...")
    dates = pd.date_range('2020-01-01', '2026-06-01', freq='B')
    np.random.seed(42)
    r = np.random.randn(len(dates)) * 0.015 + 0.0003
    close = 4000 * np.cumprod(1 + r)
    df_raw = pd.DataFrame({'close': close}, index=dates)
    price_col = 'close'


### 划分样本内 / 样本外

In [ ]:
# 样本内: 前 80% 用于参数优化
split_idx = int(len(df_raw) * 0.8)
in_sample = df_raw.iloc[:split_idx]
out_sample = df_raw.iloc[split_idx:]

print(f"样本内: {in_sample.index[0].date()} ~ {in_sample.index[-1].date()}  ({len(in_sample)} 天)")
print(f"样本外: {out_sample.index[0].date()} ~ {out_sample.index[-1].date()}  ({len(out_sample)} 天)")

close_in = in_sample[price_col]
close_out = out_sample[price_col]


## 1. 双均线策略原理

### 金叉与死叉

- **短期均线** (fast MA)：对价格变化敏感，反应快但噪音多
- **长期均线** (slow MA)：趋势更平滑，反应慢但假信号少

**金叉 (Golden Cross)**：短期均线上穿长期均线 → 买入信号  
**死叉 (Dead Cross)**：短期均线下穿长期均线 → 卖出信号

### 数学定义

$$\text{SMA}_n(t) = \frac{1}{n}\sum_{i=0}^{n-1} P_{t-i}$$

$$\text{Signal}(t) = 
\begin{cases}
1 & \text{if } \text{SMA}_{fast}(t) > \text{SMA}_{slow}(t) \\
0 & \text{otherwise}
\end{cases}$$

### 参数优化的核心问题

> 用什么样的均线窗口？5/20？10/50？20/120？答案不是唯一的——需要通过**参数搜索**找到适合当前资产和市场的组合。但搜索得越多，**过拟合风险越大**。


## 2. 单组参数示例：10日 / 50日均线

In [ ]:
# 使用 vectorbt 的 MA 指标
fast_ma = 10
slow_ma = 50

# 计算双均线交叉信号
fast = vbt.MA.run(close_in, window=fast_ma)
slow = vbt.MA.run(close_in, window=slow_ma)

# 生成入场信号：快线上穿慢线
entries = fast.ma_crossed_above(slow)
exits = fast.ma_crossed_below(slow)

# 构建投资组合
pf = vbt.Portfolio.from_signals(
    close_in, entries, exits,
    freq='1D',
    init_cash=100000,
    fees=0.001,       # 0.1% 手续费
    slippage=0.001    # 0.1% 滑点
)

print(f"初始资金: ¥100,000")
print(f"最终资金: ¥{pf.value()[-1]:,.0f}")
print(f"总收益率: {pf.total_return():.2%}")
print(f"年化收益率: {pf.annualized_return():.2%}")
print(f"夏普比: {pf.sharpe_ratio():.2f}")
print(f"最大回撤: {pf.max_drawdown():.2%}")
print(f"胜率: {pf.trades.win_rate():.2%}")
print(f"交易次数: {len(pf.trades)}")


### 绘制信号与资金曲线

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                           gridspec_kw={'height_ratios': [3, 1, 1]})

# 第一子图：价格 + 均线 + 买卖点
ax1 = axes[0]
ax1.plot(close_in.index, close_in, color='black', alpha=0.5, lw=0.8, label='沪深300')
ax1.plot(fast.ma.index, fast.ma, color='blue', lw=1, label=f'{fast_ma}日MA')
ax1.plot(slow.ma.index, slow.ma, color='red', lw=1, label=f'{slow_ma}日MA')

# 标注买卖点
entry_dates = entries.index[entries]
exit_dates = exits.index[exits]
entry_prices = close_in.loc[entry_dates] if len(entry_dates) > 0 else pd.Series(dtype=float)
exit_prices = close_in.loc[exit_dates] if len(exit_dates) > 0 else pd.Series(dtype=float)

ax1.scatter(entry_dates, entry_prices, marker='^', color='green', s=40,
            alpha=0.8, label='买入(金叉)', zorder=5)
ax1.scatter(exit_dates, exit_prices, marker='v', color='red', s=40,
            alpha=0.8, label='卖出(死叉)', zorder=5)

ax1.set_ylabel('价格')
ax1.legend(loc='upper left', fontsize=8)
ax1.set_title(f'双均线策略 ({fast_ma}/{slow_ma}) — 样本内', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 第二子图：资金曲线
ax2 = axes[1]
ax2.plot(pf.value().index, pf.value(), color='darkgreen', lw=1)
ax2.axhline(y=100000, color='gray', linestyle='--', alpha=0.5, label='初始资金')
ax2.set_ylabel('资金')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)

# 第三子图：回撤
ax3 = axes[2]
drawdown = pf.drawdown()
ax3.fill_between(drawdown.index, 0, drawdown * 100, color='red', alpha=0.3)
ax3.plot(drawdown.index, drawdown * 100, color='red', lw=0.5)
ax3.set_ylabel('回撤 (%)')
ax3.set_xlabel('日期')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


**观察要点**：
- 金叉买入、死叉卖出的信号是否及时？
- 震荡市中假信号多不多？
- 最大回撤发生在什么时期？


## 3. 参数网格搜索

现在用 vectorbt 的 `from_signals` 对多组 `(fast_window, slow_window)` 组合做批量回测。

### 搜索空间
- 快速均线窗口：5, 10, 15, 20, 25, 30, 40, 50
- 慢速均线窗口：20, 30, 40, 50, 60, 80, 100, 120

约束：`fast < slow`（短期均线窗口必须小于长期均线窗口）


In [ ]:
import itertools

# 定义参数网格
fast_windows = np.arange(5, 55, 5)
slow_windows = np.arange(20, 130, 10)

# 生成所有 (fast, slow) 组合，满足 fast < slow
param_combos = [(f, s) for f, s in itertools.product(fast_windows, slow_windows) if f < s]
print(f"总参数组合数: {len(param_combos)}")
print(f"示例: {param_combos[:6]}")


In [ ]:
# 用 vectorbt 批量计算各组参数的信号和绩效
results = []

for fast_w, slow_w in param_combos:
    fast_ma = vbt.MA.run(close_in, window=fast_w)
    slow_ma = vbt.MA.run(close_in, window=slow_w)
    
    entries = fast_ma.ma_crossed_above(slow_ma)
    exits = fast_ma.ma_crossed_below(slow_ma)
    
    pf = vbt.Portfolio.from_signals(
        close_in, entries, exits,
        freq='1D', init_cash=100000,
        fees=0.001, slippage=0.001
    )
    
    results.append({
        'fast': fast_w,
        'slow': slow_w,
        'total_return': pf.total_return(),
        'ann_return': pf.annualized_return(),
        'sharpe': pf.sharpe_ratio(),
        'max_dd': pf.max_drawdown(),
        'n_trades': len(pf.trades),
        'win_rate': pf.trades.win_rate(),
    })

df_results = pd.DataFrame(results)
print(f"回测完成，共 {len(df_results)} 组参数")
df_results.head(10)


### 按夏普比排序 — 找出最优参数

In [ ]:
# 样本内最优参数
df_sorted = df_results.sort_values('sharpe', ascending=False)
print("样本内 TOP 10 参数组合 (按夏普比):")
df_sorted[['fast', 'slow', 'sharpe', 'ann_return', 'max_dd', 'win_rate', 'n_trades']].head(10)


## 4. 夏普比热力图

In [ ]:
# 构建热力图矩阵
heatmap_data = df_results.pivot_table(
    index='slow', columns='fast', values='sharpe'
)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(heatmap_data.values, aspect='auto', cmap='RdYlGn', origin='lower')

# 标注
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)

# 标注最优
best = df_sorted.iloc[0]
best_col = list(heatmap_data.columns).index(best['fast'])
best_row = list(heatmap_data.index).index(best['slow'])
ax.plot(best_col, best_row, marker='*', color='blue', markersize=20, 
        markeredgewidth=1.5, markeredgecolor='white')

ax.set_xlabel('快速均线窗口 (天)', fontsize=12)
ax.set_ylabel('慢速均线窗口 (天)', fontsize=12)
ax.set_title('双均线策略 — 夏普比热力图 (样本内)', fontsize=14, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('夏普比', fontsize=11)

plt.tight_layout()
plt.show()

print(f"★ 样本内最优: fast={best['fast']}, slow={best['slow']}, "
      f"Sharpe={best['sharpe']:.3f}, 年化={best['ann_return']:.2%}")


**热力图解读**：
- 颜色越绿 → 夏普比越高 → 参数组合越"好"（样本内）
- 但注意：这只是在样本内数据上表现好，**不代表样本外也好**
- 如果热力图呈现随机斑块分布（无明显趋势），说明参数不稳定


## 5. 样本外验证

> **核心原则**：参数优化必须在样本内完成，然后用**完全未见过**的样本外数据评估。如果在同一条数据上优化和评估，拿到的一定是过拟合结果。


In [ ]:
# 对每组合格参数，在样本外评估
out_results = []

for _, row in df_results.iterrows():
    fast_w = int(row['fast'])
    slow_w = int(row['slow'])
    
    fast_ma = vbt.MA.run(close_out, window=fast_w)
    slow_ma = vbt.MA.run(close_out, window=slow_w)
    
    entries = fast_ma.ma_crossed_above(slow_ma)
    exits = fast_ma.ma_crossed_below(slow_ma)
    
    pf = vbt.Portfolio.from_signals(
        close_out, entries, exits, 
        freq='1D', init_cash=100000,
        fees=0.001, slippage=0.001
    )
    
    out_results.append({
        'fast': fast_w, 'slow': slow_w,
        'sharpe_out': pf.sharpe_ratio(),
        'ann_return_out': pf.annualized_return(),
        'max_dd_out': pf.max_drawdown(),
    })

df_out = pd.DataFrame(out_results)
df_compare = df_results[['fast', 'slow', 'sharpe']].copy()
df_compare = df_compare.merge(df_out, on=['fast', 'slow'])
df_compare.columns = ['fast', 'slow', 'sharpe_in', 'sharpe_out', 'ann_return_out', 'max_dd_out']

df_compare.head(8)


In [ ]:
# 样本内最优在样本外的表现
best_in = df_compare.loc[df_compare['sharpe_in'].idxmax()]
print(f"样本内最优参数: ({int(best_in['fast'])}, {int(best_in['slow'])})")
print(f"  样本内夏普: {best_in['sharpe_in']:.3f}")
print(f"  样本外夏普: {best_in['sharpe_out']:.3f}")
print(f"  衰减幅度:   {(1 - best_in['sharpe_out']/best_in['sharpe_in'])*100:.1f}%")
print()

# 样本外最优
best_out = df_compare.loc[df_compare['sharpe_out'].idxmax()]
print(f"样本外最优参数: ({int(best_out['fast'])}, {int(best_out['slow'])})")
print(f"  样本内夏普: {best_out['sharpe_in']:.3f}")
print(f"  样本外夏普: {best_out['sharpe_out']:.3f}")


### 样本内 vs 样本外散点图

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(df_compare['sharpe_in'], df_compare['sharpe_out'],
           alpha=0.5, s=20)

# 45度参考线
lims = [
    min(df_compare['sharpe_in'].min(), df_compare['sharpe_out'].min()),
    max(df_compare['sharpe_in'].max(), df_compare['sharpe_out'].max())
]
ax.plot(lims, lims, 'k--', alpha=0.3, label='y=x (完美一致)')
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)

# 标注最优
ax.scatter(best_in['sharpe_in'], best_in['sharpe_out'],
           marker='*', s=200, color='red', edgecolors='white',
           zorder=5, label=f'样本内最优({int(best_in["fast"])},{int(best_in["slow"])})')
ax.scatter(best_out['sharpe_in'], best_out['sharpe_out'],
           marker='D', s=100, color='blue', edgecolors='white',
           zorder=5, label=f'样本外最优({int(best_out["fast"])},{int(best_out["slow"])})')

ax.set_xlabel('样本内夏普比', fontsize=12)
ax.set_ylabel('样本外夏普比', fontsize=12)
ax.set_title('样本内 vs 样本外 — 夏普比衰减', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()


**如果大多数点落在 45° 线以下** → 样本外表现系统性差于样本内 → **过拟合证据**。

这几乎是必然的——策略在样本内被"调优"过，样本外自然会衰减。关键在于衰减幅度是否可接受。


## 6. 参数敏感性与过拟合

### 6.1 参数越多，过拟合风险越大

我们从 1 个参数（fast window，slow 固定）到 2 个参数（fast + slow 都搜索），搜索空间越大，找到"纯幸运"的好结果概率越高。

### 6.2 过拟合信号

- 样本内夏普比远高于样本外（衰减 > 50%）
- 最优参数是"孤岛"——相邻参数表现突然变差
- 最优参数在不同时间段不稳定

### 6.3 防御方法

| 方法 | 说明 |
|------|------|
| 样本内/外划分 | 最基础的防御 |
| 交叉验证 | 多段样本内轮换，取平均 |
| 惩罚复杂度 | AIC/BIC 等信息准则 |
| 参数鲁棒性检验 | 最优参数邻域的表现是否稳定 |
| 少即是多 | 参数越少越好，2-3 个是安全上限 |


In [ ]:
# 检查最优参数的邻域稳定性
best_fast = int(best_in['fast'])
best_slow = int(best_in['slow'])

neighbors = df_results[
    (df_results['fast'].between(best_fast - 5, best_fast + 5)) &
    (df_results['slow'].between(best_slow - 10, best_slow + 10))
]

print(f"最优参数 ({best_fast}, {best_slow}) 附近 {len(neighbors)} 个邻居：")
print(f"  夏普比均值: {neighbors['sharpe'].mean():.3f}")
print(f"  夏普比标准差: {neighbors['sharpe'].std():.3f}")
print(f"  夏普比范围: [{neighbors['sharpe'].min():.3f}, {neighbors['sharpe'].max():.3f}]")
print()
if neighbors['sharpe'].std() < 0.3:
    print("✓ 邻域夏普比稳定 → 参数不太可能是数据挖掘产物")
else:
    print("⚠ 邻域夏普比波动大 → 存在过拟合迹象，需要进一步检验")


## 7. 最终策略：样本外最优参数

选定样本外表现最好的参数，在整个数据集上回测作为最终评估。


In [ ]:
final_fast = int(best_out['fast'])
final_slow = int(best_out['slow'])

# 全数据集回测
price_full = df_raw[price_col]
fast_ma_full = vbt.MA.run(price_full, window=final_fast)
slow_ma_full = vbt.MA.run(price_full, window=final_slow)

entries_full = fast_ma_full.ma_crossed_above(slow_ma_full)
exits_full = fast_ma_full.ma_crossed_below(slow_ma_full)

pf_full = vbt.Portfolio.from_signals(
    price_full, entries_full, exits_full,
    freq='1D', init_cash=100000,
    fees=0.001, slippage=0.001
)

print("=" * 50)
print(f"最终策略: 双均线 ({final_fast}/{final_slow})")
print("=" * 50)
print(f"初始资金:     ¥{100000:,.0f}")
print(f"最终资金:     ¥{pf_full.value()[-1]:,.0f}")
print(f"总收益率:     {pf_full.total_return():.2%}")
print(f"年化收益率:   {pf_full.annualized_return():.2%}")
print(f"夏普比:       {pf_full.sharpe_ratio():.2f}")
print(f"最大回撤:     {pf_full.max_drawdown():.2%}")
print(f"Calmar 比:    {pf_full.calmar_ratio():.2f}")
print(f"交易次数:     {len(pf_full.trades)}")
print(f"胜率:         {pf_full.trades.win_rate():.2%}")
print(f"盈亏比:       {pf_full.trades.expectancy():.2f}")
print()

# 与买入持有对比
buyhold_return = (price_full.iloc[-1] / price_full.iloc[0] - 1)
buyhold_ann = (1 + buyhold_return) ** (252 / len(price_full)) - 1
print(f"【基准: 买入持有】")
print(f"  总收益率:   {buyhold_return:.2%}")
print(f"  年化收益率: {buyhold_ann:.2%}")
print(f"  策略超额:   {pf_full.annualized_return() - buyhold_ann:.2%}")


In [ ]:
# 全数据集资金曲线
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                           gridspec_kw={'height_ratios': [3, 1, 1]})

ax1 = axes[0]
ax1.plot(pf_full.value().index, pf_full.value(), color='darkgreen', lw=1.2, label='策略资金')
# 买入持有对比
bh_curve = price_full / price_full.iloc[0] * 100000
ax1.plot(bh_curve.index, bh_curve, color='gray', lw=0.8, alpha=0.6, 
         linestyle='--', label='买入持有')
ax1.axhline(y=100000, color='black', linestyle=':', alpha=0.3)
ax1.set_ylabel('资金 (¥)')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_title(f'最终策略 ({final_fast}/{final_slow} 均线) — 全数据集', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.2)

# 回撤
drawdown_full = pf_full.drawdown()
ax2 = axes[1]
ax2.fill_between(drawdown_full.index, 0, drawdown_full * 100, color='red', alpha=0.3)
ax2.plot(drawdown_full.index, drawdown_full * 100, color='red', lw=0.5)
ax2.set_ylabel('回撤 (%)')
ax2.grid(True, alpha=0.2)

# 交易分布
trades = pf_full.trades.records_readable
if len(trades) > 0:
    pnl_pct = (trades['Exit Price'].values - trades['Entry Price'].values) / trades['Entry Price'].values * 100
    ax3 = axes[2]
    colors_pnl = ['green' if x > 0 else 'red' for x in pnl_pct]
    ax3.bar(range(len(pnl_pct)), pnl_pct, color=colors_pnl, alpha=0.7)
    ax3.axhline(y=0, color='black', lw=0.5)
    ax3.set_ylabel('收益率 (%)')
    ax3.set_xlabel('交易序号')
    ax3.set_title(f'单笔交易收益率分布 ({len(pnl_pct)} 笔)', fontsize=10)

plt.tight_layout()
plt.show()


## 8. 小结

### 核心要点

1. **双均线策略**是最基础的量化策略，金叉买入、死叉卖出
2. **参数优化**在样本内完成，**评估在样本外**——这是铁律
3. **夏普比热力图**可以直观展示参数敏感性和稳定性
4. **样本内到样本外的衰减是必然的**，关键在于衰减幅度是否可控
5. 策略必须考虑**手续费和滑点**，否则回测结果不可信

### 策略局限

- 震荡市中双均线会产生大量假信号（反复穿越）
- 参数固定不变，无法适应市场风格切换
- 缺少仓位管理和止损机制（后续模块会补充）

### 验收标准

- [ ] 理解金叉/死叉的逻辑和数学定义
- [ ] 能用 vectorbt 实现双均线策略并回测
- [ ] 能对参数网格做批量搜索并绘制夏普比热力图
- [ ] 能区分样本内和样本外，理解为什么必须分开
- [ ] 能识别参数过拟合的典型信号
- [ ] 策略含手续费和滑点，不做裸回测
- [ ] 对比了买入持有基准

---

*下一个主题：序号23 — 动量与均值回归策略*
